# Sandbox Dashboard Parity Notebook

This notebook mirrors the dashboard logic in `app.py` step by step.

It is segmented into two parts that match the Streamlit pages:
1. **Product Analysis**
2. **Infrastructure & Cost Analysis**

For each part, we move from key metrics to prototype charts, while exposing the important intermediate calculations used by the dashboard.

In [ ]:
import math

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

import app

# Load exactly the same datasets used by the dashboard.
hourly_usage, infrastructure_costs, research_requests, users = app.load_datasets_from_zip()

print("Loaded shapes:")
print("hourly_usage:", hourly_usage.shape)
print("infrastructure_costs:", infrastructure_costs.shape)
print("research_requests:", research_requests.shape)
print("users:", users.shape)

## Part 1 - Product Analysis

This section mirrors the product page in the dashboard, from KPI calculations to prototype visuals.

In [ ]:
# Part 1 - Product Analysis
# We replicate product dashboard logic from metrics to chart-ready tables.
# The next cells compute KPIs and then render one prototype chart per cell.

In [ ]:
# Step 1: Product top-metrics calculations
# We explicitly compute the core values shown in dashboard metrics.

lifecycle, joined_users_count = app._build_hourly_lifecycle(users, hourly_usage)
research_first_count = int(lifecycle["first_source"].eq("research").sum())
active_joined_users_count = int(lifecycle["user_id"].nunique())
acquisition_pct = (
    100.0 * research_first_count / active_joined_users_count if active_joined_users_count else 0.0
)

rr = app._lowercase_columns(research_requests).copy()
status_norm = rr["status"].astype(str).str.strip().str.lower()
is_cancelled = app._is_cancelled_status(status_norm)

success_count = int(status_norm.eq("success").sum())
cancelled_count = int(is_cancelled.sum())
total_count = int(len(status_norm))

success_rate_pct = 100.0 * success_count / total_count if total_count else 0.0
cancellation_rate_pct = 100.0 * cancelled_count / total_count if total_count else 0.0

total_request_cost = float(pd.to_numeric(rr["request_cost"], errors="coerce").fillna(0.0).sum())

stream_flag = app._is_true_stream(rr["stream"])
stream_total = int(stream_flag.sum())
non_stream_total = int((~stream_flag).sum())
stream_cancelled = int((is_cancelled & stream_flag).sum())
non_stream_cancelled = int((is_cancelled & ~stream_flag).sum())
stream_cancel_rate_pct = 100.0 * stream_cancelled / stream_total if stream_total else 0.0
non_stream_cancel_rate_pct = 100.0 * non_stream_cancelled / non_stream_total if non_stream_total else 0.0

product_metrics = {
    "acquisition_pct": acquisition_pct,
    "total_request_cost": total_request_cost,
    "success_rate_pct": success_rate_pct,
    "cancellation_rate_pct": cancellation_rate_pct,
    "streaming_cancellation_rate_pct": stream_cancel_rate_pct,
    "non_streaming_cancellation_rate_pct": non_stream_cancel_rate_pct,
}

print("Product metrics:")
for k, v in product_metrics.items():
    if "cost" in k:
        print(f"- {k}: ${v:,.2f}")
    else:
        print(f"- {k}: {v:,.2f}%")

In [ ]:
# Step 2: Product chart input tables and economics summaries
# These are the prepared datasets used across multiple product charts.

prepared_cancel = app._prepare_cancellation_chart_data(research_requests)
if prepared_cancel is None:
    raise ValueError("Cancellation chart inputs unavailable")

wait_effect, inefficiency_long, billing_dist, stream_metrics = prepared_cancel

q2_data = app._prepare_user_and_cost_breakdowns(users, research_requests)
if q2_data is None:
    raise ValueError("User/cost breakdown inputs unavailable")

user_dist, request_cost_dist, cost_by_model_user, economics_summary = q2_data

print("wait_effect")
display(wait_effect)
print("inefficiency_long")
display(inefficiency_long)
print("billing_dist")
display(billing_dist)
print("user_dist")
display(user_dist)
print("request_cost_dist (sample)")
display(request_cost_dist.head())
print("cost_by_model_user")
display(cost_by_model_user)
print("economics_summary")
print(economics_summary)

In [ ]:
# Part 1 chart section
# Each of the following cells renders one prototype chart.
# This keeps chart logic isolated and easy to validate.

#### Chart 1 - Abandonment rate by first platform feature

This reproduces the logic of abandonment by first request type and highlights the Research bar.

In [ ]:
no_return_df = app._single_row_no_return_by_first_request(lifecycle).copy()
no_return_df["first_request_label_display"] = no_return_df["first_request_label"].astype(str).str.capitalize()
label_order = no_return_df["first_request_label_display"].tolist()
bar_colors = ["#E45756" if x.lower() == "research" else "#4C78A8" for x in no_return_df["first_request_label_display"]]

fig_retention = px.bar(
    no_return_df,
    x="first_request_label_display",
    y="pct_single_row",
    text=no_return_df["pct_single_row"].map(lambda x: f"{x:.2f}%"),
    title="Prototype - Abandonment rate by platform feature",
    category_orders={"first_request_label_display": label_order},
)
fig_retention.update_traces(marker_color=bar_colors, textposition="outside", cliponaxis=False)
fig_retention.show()

#### Chart 2 - Traffic concentration (Pareto)

Shows cumulative share of traffic as cumulative share of users grows.

In [ ]:
pareto, y_at_5 = app._prepare_pareto(research_requests)
fig_pareto = px.line(
    pareto,
    x="cum_users_pct",
    y="cum_requests_pct",
    title="Prototype - Traffic share over users share",
)
fig_pareto.add_vline(x=5.0, line_dash="dash")
fig_pareto.add_hline(y=y_at_5, line_dash="dash")
fig_pareto.show()
print(f"Top 5% users generate about {y_at_5:.2f}% of requests")

#### Chart 3 - User base split

Free vs paying user distribution.

In [ ]:
fig_user_dist = px.pie(
    user_dist,
    values="users",
    names="user_type",
    hole=0.5,
    title="Prototype - Free vs paying users",
)
fig_user_dist.show()

#### Chart 4 - Request cost distribution by model

Validates unit cost spread for MINI vs PRO request costs.

In [ ]:
fig_cost_box = px.box(
    request_cost_dist,
    x="model",
    y="request_cost",
    color="model",
    title="Prototype - Request cost distribution by model",
    points=False,
)
fig_cost_box.show()

mini_median = float(request_cost_dist.loc[request_cost_dist["model"].eq("MINI"), "request_cost"].median())
pro_median = float(request_cost_dist.loc[request_cost_dist["model"].eq("PRO"), "request_cost"].median())
if mini_median > 0:
    print(f"Median PRO/MINI cost ratio: {pro_median / mini_median:.2f}x")

#### Chart 5 - Total request cost by model and user type

Highlights free-user spend contribution by model.

In [ ]:
fig_stacked = px.bar(
    cost_by_model_user,
    x="model",
    y="request_cost",
    color="user_type",
    barmode="stack",
    title="Prototype - Total request cost by model and user type",
)
fig_stacked.show()

print("Economics summary values used in dashboard narrative:")
print(economics_summary)

#### Chart 6 - Cancelled requests response-time histogram

Buckets cancelled requests by 30-second duration bins.

In [ ]:
cancelled_points = app._prepare_cancelled_response_times(research_requests)
max_response_time = float(cancelled_points["response_time_seconds"].max())
histogram_end = max(30.0, math.ceil(max_response_time / 30.0) * 30.0)
bin_edges = list(range(0, int(histogram_end) + 30, 30))
bucket_labels = [f"{start}-{start + 30}" for start in bin_edges[:-1]]

cancelled_points = cancelled_points.copy()
cancelled_points["duration_bucket"] = pd.cut(
    cancelled_points["response_time_seconds"],
    bins=bin_edges,
    right=False,
    include_lowest=True,
    labels=bucket_labels,
)
bucket_counts = (
    cancelled_points.groupby("duration_bucket", observed=False).size().reindex(bucket_labels, fill_value=0).reset_index(name="cancelled_requests")
)

fig_cancel_hist = go.Figure(
    data=[go.Bar(x=bucket_counts["duration_bucket"], y=bucket_counts["cancelled_requests"])]
)
fig_cancel_hist.update_layout(title="Prototype - Cancelled response time histogram")
fig_cancel_hist.show()

#### Chart 7 - Cancellation rate by wait-time threshold

Compares <90 seconds vs >=90 seconds for streaming-enabled requests.

In [ ]:
fig_wait = px.bar(
    wait_effect,
    x="duration_group",
    y="cancel_rate",
    color="duration_group",
    title="Prototype - Cancellation rate by wait time",
    text=wait_effect["cancel_rate"].map(lambda v: f"{100*v:.2f}%"),
)
fig_wait.show()
display(wait_effect)

#### Chart 8 - Technical inefficiency and cancelled billing status

Two chart cells below expose the computational inefficiency signal and the billed-vs-unbilled split for cancelled requests.

In [ ]:
fig_ineff = px.bar(
    inefficiency_long,
    x="duration_group",
    y="value",
    color="metric",
    barmode="group",
    title="Prototype - Technical inefficiency by wait time",
)
fig_ineff.show()
display(inefficiency_long)

In [ ]:
fig_billing = px.pie(
    billing_dist,
    names="billing_status",
    values="requests",
    hole=0.5,
    title="Prototype - Billing status of cancelled requests",
)
fig_billing.show()
display(billing_dist)

## Part 2 - Infrastructure & Cost Analysis

This section mirrors the infrastructure page: KPI calculations first, then one prototype chart per cell.

In [ ]:
# Step 1: Prepare infrastructure datasets and KPI inputs.
prepared_finops = app._prepare_finops_data(infrastructure_costs, hourly_usage, research_requests)
if prepared_finops is None:
    raise ValueError("FinOps data unavailable")

finops_metrics, daily_agg, monthly_agg, heatmap_data = prepared_finops

total_cost_base = finops_metrics["total_hardware_cost"] + finops_metrics["total_ai_cost"]
infra_share_pct = 100.0 * finops_metrics["total_hardware_cost"] / total_cost_base if total_cost_base else 0.0
model_share_pct = 100.0 * finops_metrics["total_ai_cost"] / total_cost_base if total_cost_base else 0.0

print("finops_metrics")
print(finops_metrics)
print({"infra_share_pct": infra_share_pct, "model_share_pct": model_share_pct})

### Growth ratios calculation (Nov 2025 -> Mar 2026)

The dashboard computes growth as end/start ratios for new users, requests, and infra spend.

In [ ]:
users_l = app._lowercase_columns(users).copy()
hourly_l = app._lowercase_columns(hourly_usage).copy()
infra_l = app._lowercase_columns(infrastructure_costs).copy()

growth_start = pd.Timestamp("2025-11-01", tz="UTC")
growth_end = pd.Timestamp("2026-03-01", tz="UTC")

users_l["created_at"] = pd.to_datetime(users_l["created_at"], errors="coerce", utc=True)
users_l = users_l.dropna(subset=["created_at"]).copy()
users_l["month"] = users_l["created_at"].dt.to_period("M").dt.to_timestamp().dt.tz_localize("UTC")
users_monthly = users_l.groupby("month").size().rename("new_users")

hourly_l["hour"] = pd.to_datetime(hourly_l["hour"], errors="coerce", utc=True)
hourly_l["request_count"] = pd.to_numeric(hourly_l["request_count"], errors="coerce").fillna(0.0)
hourly_l = hourly_l.dropna(subset=["hour"]).copy()
hourly_l["month"] = hourly_l["hour"].dt.to_period("M").dt.to_timestamp().dt.tz_localize("UTC")
req_monthly = hourly_l.groupby("month")["request_count"].sum()

infra_l["hour"] = pd.to_datetime(infra_l["hour"], errors="coerce", utc=True)
infra_cols = [c for c in infra_l.columns if c.startswith("infra_")]
for col in infra_cols:
    infra_l[col] = pd.to_numeric(infra_l[col], errors="coerce").fillna(0.0)
infra_l = infra_l.dropna(subset=["hour"]).copy()
infra_l["infra_total"] = infra_l[infra_cols].sum(axis=1)
infra_l["month"] = infra_l["hour"].dt.to_period("M").dt.to_timestamp().dt.tz_localize("UTC")
infra_monthly = infra_l.groupby("month")["infra_total"].sum()

user_start, user_end = float(users_monthly.get(growth_start, 0.0)), float(users_monthly.get(growth_end, 0.0))
req_start, req_end = float(req_monthly.get(growth_start, 0.0)), float(req_monthly.get(growth_end, 0.0))
infra_start, infra_end = float(infra_monthly.get(growth_start, 0.0)), float(infra_monthly.get(growth_end, 0.0))

def growth_ratio(start_value, end_value):
    return (end_value / start_value) if start_value > 0 else 0.0

print({
    "users_growth_ratio": growth_ratio(user_start, user_end),
    "requests_growth_ratio": growth_ratio(req_start, req_end),
    "infra_growth_ratio": growth_ratio(infra_start, infra_end),
})

### Chart 1 - Hourly total cost stability (box)

Shows distribution of total hourly cost across the period.

In [ ]:
model_cols = [c for c in infra_l.columns if c.startswith("model_")]
for col in model_cols:
    infra_l[col] = pd.to_numeric(infra_l[col], errors="coerce").fillna(0.0)
infra_l["model_total"] = infra_l[model_cols].sum(axis=1) if model_cols else 0.0
infra_l["total_hourly_cost"] = infra_l["infra_total"] + infra_l["model_total"]

fig_stability = px.box(
    infra_l[["total_hourly_cost"]].dropna(),
    y="total_hourly_cost",
    points=False,
    title="Prototype - Hourly total cost stability",
)
fig_stability.show()

### Chart 2 - Infrastructure vs model spend split

Replicates the donut split of total spend composition.

In [ ]:
budget_split = pd.DataFrame({
    "category": ["Infrastructure costs", "Model costs"],
    "cost": [finops_metrics["total_hardware_cost"], finops_metrics["total_ai_cost"]],
})
fig_donut = px.pie(budget_split, names="category", values="cost", hole=0.5, title="Prototype - Spend split")
fig_donut.show()

### Chart 3 - Requests vs total daily cost trend

Uses 7-day moving averages and computes raw daily correlation.

In [ ]:
growth_daily = daily_agg.sort_values("day").copy()
growth_daily["requests_ma7"] = growth_daily["total_requests"].rolling(window=7, min_periods=1).mean()
growth_daily["total_cost_ma7"] = growth_daily["total_daily_cost"].rolling(window=7, min_periods=1).mean()

fig_growth = make_subplots(specs=[[{"secondary_y": True}]])
fig_growth.add_trace(go.Scatter(x=growth_daily["day"], y=growth_daily["requests_ma7"], name="Requests (7d MA)"), secondary_y=False)
fig_growth.add_trace(go.Scatter(x=growth_daily["day"], y=growth_daily["total_cost_ma7"], name="Cost (7d MA)"), secondary_y=True)
fig_growth.update_layout(title="Prototype - Requests vs total cost")
fig_growth.show()

corr_daily = growth_daily[["total_requests", "total_daily_cost"]].dropna()
pearson_r_daily = float(corr_daily["total_requests"].corr(corr_daily["total_daily_cost"])) if len(corr_daily) > 1 else 0.0
print({"pearson_r_daily": pearson_r_daily})

### Chart 4 - Mean infrastructure cost heatmap

Shows weekly-hourly cyclic cost pattern.

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heatmap_data = heatmap_data.copy()
heatmap_data["day_of_week"] = pd.Categorical(heatmap_data["day_of_week"], categories=day_order, ordered=True)
heatmap_data["hour_of_day"] = pd.to_numeric(heatmap_data["hour_of_day"], errors="coerce").astype("Int64")

cost_heatmap_pivot = (
    heatmap_data.pivot_table(index="day_of_week", columns="hour_of_day", values="mean_infra_cost", aggfunc="mean")
    .reindex(day_order)
    .reindex(columns=list(range(24)))
)

fig_heatmap = px.imshow(cost_heatmap_pivot, aspect="auto", title="Prototype - Mean infra cost by day/hour")
fig_heatmap.show()

### Chart 5 - Weekend vs weekday comparison

Compares average hourly traffic and cost index values.

In [ ]:
weekend_req = hourly_l.groupby("hour", as_index=False)["request_count"].sum().rename(columns={"request_count": "total_requests"})
weekend_total_cost = infra_l.groupby("hour", as_index=False)["total_hourly_cost"].sum()
weekend_joined = weekend_req.merge(weekend_total_cost, on="hour", how="inner").dropna()
weekend_joined["day_type"] = weekend_joined["hour"].dt.day_name().apply(lambda x: "Weekend" if x in ["Saturday", "Sunday"] else "Weekday")

comparison = weekend_joined.groupby("day_type", as_index=False).agg(
    total_requests_mean=("total_requests", "mean"),
    total_cost_mean=("total_hourly_cost", "mean"),
)

weekend_row = comparison.loc[comparison["day_type"].eq("Weekend")]
weekday_row = comparison.loc[comparison["day_type"].eq("Weekday")]
weekend_req_mean = float(weekend_row["total_requests_mean"].iloc[0])
weekday_req_mean = float(weekday_row["total_requests_mean"].iloc[0])
weekend_cost_mean = float(weekend_row["total_cost_mean"].iloc[0])
weekday_cost_mean = float(weekday_row["total_cost_mean"].iloc[0])

comparison_index = pd.DataFrame({
    "metric": ["Average hourly traffic", "Average hourly traffic", "Average hourly total cost", "Average hourly total cost"],
    "day_type": ["Weekday", "Weekend", "Weekday", "Weekend"],
    "index_value": [100.0, 100.0 * weekend_req_mean / weekday_req_mean, 100.0, 100.0 * weekend_cost_mean / weekday_cost_mean],
})

fig_weekend = px.bar(comparison_index, x="metric", y="index_value", color="day_type", barmode="group", title="Prototype - Weekend vs weekday")
fig_weekend.show()
display(comparison)

### Chart 6 - Hours with vs without Research activity

Also computes the zero-traffic infra burn estimate for the research cluster.

In [ ]:
hourly_activity = app._lowercase_columns(hourly_usage).copy()
hourly_activity["hour"] = pd.to_datetime(hourly_activity["hour"], errors="coerce", utc=True)
hourly_activity["request_count"] = pd.to_numeric(hourly_activity["request_count"], errors="coerce").fillna(0.0)
hourly_activity["request_type"] = hourly_activity["request_type"].astype(str).str.strip().str.lower()
hourly_activity = hourly_activity.dropna(subset=["hour"]).copy()

research_hourly = (
    hourly_activity.loc[hourly_activity["request_type"].eq("research")]
    .groupby("hour", as_index=False)["request_count"].sum()
    .rename(columns={"request_count": "research_requests"})
)
all_hours = pd.DataFrame({"hour": hourly_activity["hour"].drop_duplicates().sort_values().to_list()})
presence = all_hours.merge(research_hourly, on="hour", how="left")
presence["research_requests"] = presence["research_requests"].fillna(0.0)
presence["activity_status"] = presence["research_requests"].apply(lambda v: "Hours with Research API activity" if v > 0 else "Hours without Research API activity")

activity_counts = presence.groupby("activity_status", as_index=False).size().rename(columns={"size": "hours_count"})
total_hours_count = int(activity_counts["hours_count"].sum())
activity_counts["hours_share_pct"] = 100.0 * activity_counts["hours_count"] / total_hours_count if total_hours_count else 0.0

fig_activity = px.bar(activity_counts, x="activity_status", y="hours_share_pct", color="activity_status", title="Prototype - Research activity presence")
fig_activity.show()

if "infra_eks_research_cluster" in infra_l.columns:
    zero_hours = presence.loc[presence["research_requests"].le(0), "hour"]
    cluster_cost = infra_l[["hour", "infra_eks_research_cluster"]].copy()
    cluster_cost["infra_eks_research_cluster"] = pd.to_numeric(cluster_cost["infra_eks_research_cluster"], errors="coerce").fillna(0.0)
    zero_traffic_burn = float(cluster_cost.merge(zero_hours.to_frame(name="hour"), on="hour", how="inner")["infra_eks_research_cluster"].sum())
    print({"zero_traffic_hours": int(len(zero_hours)), "research_cluster_zero_traffic_burn": zero_traffic_burn})